# 06b — Evaluación comparativa: base vs fine-tuned vs RAG

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Fine-tuning

---

Este notebook evalúa el impacto del fine-tuning comparando tres modelos sobre las 5 preguntas
de ground truth del pipeline RAG: el modelo base Qwen2.5-1.5B sin especializar, el modelo
fine-tuneado con QLoRA y el pipeline RAG completo con Qwen3-8B como referencia de producción.

In [ ]:
"""
Notebook: 06b_evaluacion.ipynb

Objetivo:
    Evaluación cuantitativa comparando tres aproximaciones para responder
    preguntas sobre química organometálica sobre las 5 preguntas de ground truth:

    Modelos evaluados:
      1. Base    : Qwen/Qwen2.5-1.5B-Instruct
                   Sin fine-tuning ni contexto externo — línea de referencia base
      2. Tuneado : Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora
                   Fine-tuned con QLoRA sobre chemistry-organometallic-qa
                   Conocimiento del dominio internalizado en los pesos LoRA
      3. RAG     : Qwen/Qwen3-8B + ChromaDB + BGE-M3
                   Pipeline RAG completo del proyecto — referencia de producción
                   Conocimiento del dominio inyectado como contexto externo

    Esta comparación responde a la pregunta clave del proyecto:
    ¿Cuánto mejora el fine-tuning respecto al modelo base? ¿Se acerca
    al rendimiento del RAG? ¿Son estrategias complementarias o sustitutivas?

    Este notebook cubre:
      1. Configuración del entorno
      2. Instalación de dependencias
      3. Carga de los tres modelos
      4. Generación de respuestas con los tres modelos
      5. Evaluación cuantitativa con similitud semántica BGE-M3
      6. Comparación de resultados y conclusiones
      7. Resumen final

    Metodología de evaluación:
      Se usa similitud coseno entre embeddings BGE-M3 de las respuestas
      generadas y el ground truth — la misma métrica usada en
      02b_chunking_eval.ipynb para evaluar estrategias de chunking,
      garantizando coherencia metodológica en todo el proyecto.

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Siguiente paso:
    Publicación del modelo fine-tuneado en HF Hub como demo interactivo

Autor:   Jesús Rodríguez
Fecha:   2026-05-20
Versión: 1.0.0
"""

'\nNotebook: 06b_evaluacion.ipynb\n\nObjetivo:\n    Evaluación cuantitativa comparando tres aproximaciones para responder\n    preguntas sobre química organometálica sobre las 5 preguntas de ground truth:\n\n    Modelos evaluados:\n      1. Base    : Qwen/Qwen2.5-1.5B-Instruct\n                   Sin fine-tuning ni contexto externo — línea de referencia base\n      2. Tuneado : Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora\n                   Fine-tuned con QLoRA sobre chemistry-organometallic-qa\n                   Conocimiento del dominio internalizado en los pesos LoRA\n      3. RAG     : Qwen/Qwen3-8B + ChromaDB + BGE-M3\n                   Pipeline RAG completo del proyecto — referencia de producción\n                   Conocimiento del dominio inyectado como contexto externo\n\n    Esta comparación responde a la pregunta clave del proyecto:\n    ¿Cuánto mejora el fine-tuning respecto al modelo base? ¿Se acerca\n    al rendimiento del RAG? ¿Son estrategias complementarias o sustit

## 1. Configuración del entorno

In [ ]:
# Librería estándar
import json
import os
import time
from pathlib import Path

# Third-party — se importan tras la instalación en Sección 2
import sys
print(f'Python version: {sys.version}')

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
# Verificar disponibilidad de GPU
# BGE-M3 y la inferencia con el modelo fine-tuneado son más rápidos en GPU
import subprocess
import torch

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'GPU detectada: {result.stdout.strip()}')
else:
    print('GPU no detectada — la evaluación funcionará en CPU pero será más lenta')

print(f'Dispositivo activo: {"CUDA" if torch.cuda.is_available() else "CPU"}')

GPU detectada: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Dispositivo activo: CUDA


In [ ]:
# Kaggle usa /kaggle/working/ como directorio de trabajo persistente
# No requiere montaje de Drive — los resultados se guardan localmente
DIR_FINETUNE = Path('/kaggle/working')
DIR_FINETUNE.mkdir(parents=True, exist_ok=True)

print(f'DIR_FINETUNE  : {DIR_FINETUNE}')

DIR_FINETUNE  : /kaggle/working


In [ ]:
# En Kaggle los secretos se añaden en Settings → Add-ons → Secrets
# y se acceden con kaggle_secrets.UserSecretsClient
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN cargado desde secretos de Kaggle.')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')
    print('HF_TOKEN cargado desde variable de entorno.')

assert HF_TOKEN, 'HF_TOKEN no encontrado. Añádelo en Kaggle → Add-ons → Secrets.'

assert HF_TOKEN, 'HF_TOKEN no encontrado. Añádelo en Colab → Secrets.'

# Identificadores de los tres modelos a comparar
MODELO_BASE    = 'Qwen/Qwen2.5-1.5B-Instruct'                           # Modelo base sin especializar
MODELO_TUNEADO = 'Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora'        # Fine-tuned con QLoRA
MODELO_RAG     = 'Qwen/Qwen3-8B'                                        # Modelo del pipeline RAG
MODELO_EMBED   = 'BAAI/bge-m3'                                          # Embeddings para evaluación

# System prompt — idéntico al usado en el fine-tuning
# para garantizar coherencia en la evaluación
SYSTEM_PROMPT = (
    'You are a specialized scientific assistant in organometallic chemistry '
    'and medicinal inorganic chemistry. Answer questions accurately and '
    'concisely based on your knowledge of NHC metal complexes.'
)

print(f'Modelo base    : {MODELO_BASE}')
print(f'Modelo tuneado : {MODELO_TUNEADO}')
print(f'Modelo RAG     : {MODELO_RAG}')
print(f'Modelo embed   : {MODELO_EMBED}')

HF_TOKEN cargado desde secretos de Kaggle.
Modelo base    : Qwen/Qwen2.5-1.5B-Instruct
Modelo tuneado : Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora
Modelo RAG     : Qwen/Qwen3-8B
Modelo embed   : BAAI/bge-m3


In [ ]:
# Las 5 preguntas de ground truth del pipeline RAG
# Las preguntas son las mismas usadas en 03_recuperacion.ipynb y 05_dataset.ipynb
# Las respuestas coinciden con 05_dataset.ipynb — respuestas detalladas verificadas
# manualmente, coherentes con el dataset de entrenamiento del fine-tuning
GROUND_TRUTH_QA = [
    {
        'pregunta': 'What metals are used in the NHC complexes studied?',
        'ground_truth': (
            'The NHC complexes studied contain palladium (Pd), platinum (Pt) '
            'and gold (Au) as metal centers. Specifically, Pd(II), Pt(II) and '
            'Au(III) tetracarbene complexes were synthesized using ethylene-bridged '
            'tetradentate NHC ligands.'
        )
    },
    {
        'pregunta': 'What is the effect of the complexes on cisplatin-resistant neuroblastoma cells?',
        'ground_truth': (
            'AuL9 induces apoptosis in cisplatin-resistant SK-N-AS neuroblastoma '
            'cells in vitro via the mitochondrial and ROS pathway. The complex '
            'overcomes cisplatin resistance, suggesting that procaspase-8 plays '
            'a minor role in AuL9-induced apoptosis.'
        )
    },
    {
        'pregunta': 'What analytical techniques were used to characterize the compounds?',
        'ground_truth': (
            'The compounds were characterized by NMR spectroscopy (1H, 13C, 19F), '
            'elemental analysis (C/H/N) at the Microanalytical Laboratory of TUM, '
            'and electrospray ionization mass spectrometry (ESI-MS and HR-ESI-MS) '
            'on a Thermo Fisher Orbitrap. Single-crystal X-ray diffraction (SC-XRD) '
            'was used for structural characterization of PdL3, PtL3 and PdL9.'
        )
    },
    {
        'pregunta': 'What is the role of the ethylene bridge in the tetra-NHC ligand design?',
        'ground_truth': (
            'The ethylene bridge connects the NHC units to form cyclic tetradentate '
            'ligands. It introduces a +I inductive effect that increases electron '
            'density on the carbene carbon, leading to upfield shifts in NMR. The '
            'bridge geometry forces the ligand into a macrocyclic structure that '
            'enables tetracarbene coordination to a single metal center.'
        )
    },
    {
        'pregunta': 'How do the cytotoxicity results of the Au(III) complexes compare to cisplatin?',
        'ground_truth': (
            'AuL9 shows higher cytotoxicity than cisplatin in cisplatin-resistant '
            'SK-N-AS neuroblastoma cells. While cisplatin fails in resistant cells, '
            'AuL9 induces significant apoptosis and inhibits proliferation in a '
            'dose-dependent manner, with nearly 100% inhibition at 50 μM.'
        )
    },
]

print(f'Preguntas de evaluación: {len(GROUND_TRUTH_QA)}')

Preguntas de evaluación: 5


## 2. Instalación de dependencias

In [ ]:
# Dependencias para inferencia QLoRA, evaluación semántica y llamadas a HF Inference API
!pip install transformers peft bitsandbytes FlagEmbedding accelerate huggingface_hub --quiet
print('Dependencias instaladas.')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.9 MB/s eta 0:00:00
Dependencias instaladas.


In [ ]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from FlagEmbedding import BGEM3FlagModel
from huggingface_hub import InferenceClient

print(f'torch version  : {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

torch version  : 2.10.0+cu128
CUDA disponible: True


## 3. Carga de modelos

In [ ]:
# Configuración de quantización 4-bit — idéntica a la usada en el fine-tuning
# Garantiza que el modelo base se carga en las mismas condiciones
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Configuración QLoRA 4-bit NF4 lista.')

Configuración QLoRA 4-bit NF4 lista.


In [ ]:
# Cargar tokenizador — compartido por ambos modelos
# El modelo fine-tuneado usa el mismo tokenizador del modelo base
tokenizer = AutoTokenizer.from_pretrained(
    MODELO_BASE,
    token=HF_TOKEN,
    padding_side='left',  # left padding para inferencia con modelos causales
)
tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizador cargado: {MODELO_BASE}')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizador cargado: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
# Cargar modelo base — sin adaptadores LoRA
# Se usa como línea de referencia para medir el impacto del fine-tuning
print('Cargando modelo base...')
modelo_base = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
modelo_base.eval()
print(f'Modelo base cargado: {MODELO_BASE}')

Cargando modelo base...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo base cargado: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
# Cargar modelo fine-tuneado — modelo base + adaptadores LoRA
# PeftModel aplica los adaptadores sobre el modelo base cargado
# sin modificar los pesos originales
print('Cargando modelo fine-tuneado...')

# is_trainable=False — modo inferencia
# Al pasar modelo_base ya cargado se ignora base_model_name_or_path
# del adapter_config.json que apunta a ruta local de Colab
modelo_tuneado = PeftModel.from_pretrained(
    modelo_base,
    MODELO_TUNEADO,
    token=HF_TOKEN,
    is_trainable=False,
)

modelo_tuneado.eval()
print(f'Modelo fine-tuneado cargado: {MODELO_TUNEADO}')

Cargando modelo fine-tuneado...


adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

Modelo fine-tuneado cargado: Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora


In [ ]:
# Cargar BGE-M3 para evaluación semántica
# La misma métrica usada en 02b_chunking_eval.ipynb
# garantiza coherencia metodológica en todo el proyecto
print('Cargando BGE-M3 para evaluación semántica...')
modelo_embed = BGEM3FlagModel(
    MODELO_EMBED,
    use_fp16=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print('BGE-M3 cargado.')

Cargando BGE-M3 para evaluación semántica...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 cargado.


In [ ]:
# Inicializar cliente para Qwen3-8B vía HF Inference API
# El modelo RAG no se carga localmente — se llama como servicio externo
# igual que en el pipeline RAG del proyecto (04_generacion.ipynb)
cliente_rag = InferenceClient(
    provider='auto',
    api_key=HF_TOKEN,
)
print(f'Cliente RAG inicializado para: {MODELO_RAG}')

Cliente RAG inicializado para: Qwen/Qwen3-8B


## 4. Generación de respuestas con los tres modelos

In [ ]:
def generar_respuesta_local(modelo, pregunta: str, max_new_tokens: int = 300) -> str:
    """Genera una respuesta usando un modelo cargado localmente.

    Usado tanto para el modelo base como para el fine-tuneado.
    Aplica el chat template de Qwen2.5 automáticamente mediante
    el tokenizador para garantizar el formato correcto de entrada.

    Args:
        modelo: modelo cargado con from_pretrained o PeftModel.
        pregunta: pregunta en lenguaje natural.
        max_new_tokens: máximo de tokens a generar en la respuesta.

    Returns:
        Respuesta generada como string.
    """
    mensajes = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': pregunta},
    ]

    # Aplicar chat template — convierte mensajes a tokens con formato correcto
    inputs = tokenizer.apply_chat_template(
        mensajes,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    ).to(modelo.device)
    input_ids = inputs['input_ids']

    with torch.no_grad():
        output_ids = modelo.generate(
            input_ids,
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            do_sample=False,         # Greedy decoding — respuestas deterministas
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decodificar solo los tokens generados — excluir el prompt de entrada
    tokens_generados = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(tokens_generados, skip_special_tokens=True).strip()


def generar_respuesta_rag(pregunta: str) -> str:
    """Genera una respuesta usando Qwen3-8B vía HF Inference API.

    El modelo RAG no se carga localmente — se llama como servicio externo
    usando InferenceClient, igual que en el pipeline RAG del proyecto.
    Usa /no_think para desactivar el modo de razonamiento de Qwen3.

    Args:
        pregunta: pregunta en lenguaje natural.

    Returns:
        Respuesta generada como string.
    """
    import re
    respuesta = cliente_rag.chat.completions.create(
        model=MODELO_RAG,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': pregunta + ' /no_think'},
        ],
        max_tokens=300,
        temperature=0.7,
        top_p=0.8,
    )
    texto = respuesta.choices[0].message.content
    # Eliminar bloque <think> si /no_think no fue respetado
    return re.sub(r'<think>.*?</think>\s*', '', texto, flags=re.DOTALL).strip()


print('Funciones de generación definidas.')

Funciones de generación definidas.


In [ ]:
import time

# Generar respuestas con los tres modelos para cada pregunta
# Se persiste en disco para no regenerar si el runtime se reinicia
RUTA_RESPUESTAS = DIR_FINETUNE / 'respuestas_evaluacion.json'

if RUTA_RESPUESTAS.exists():
    with open(RUTA_RESPUESTAS, 'r', encoding='utf-8') as f:
        resultados = json.load(f)
    print(f'Respuestas cargadas desde disco: {len(resultados)} preguntas')
else:
    resultados = []

    for i, qa in enumerate(GROUND_TRUTH_QA):
        print(f'\n[{i+1}/5] {qa["pregunta"][:70]}...')

        # Modelo base
        t0 = time.time()
        resp_base = generar_respuesta_local(modelo_base, qa['pregunta'])
        t_base = time.time() - t0
        print(f'  Base    ({t_base:.1f}s): {resp_base[:80]}...')

        # Modelo fine-tuneado
        t0 = time.time()
        resp_tuneado = generar_respuesta_local(modelo_tuneado, qa['pregunta'])
        t_tuneado = time.time() - t0
        print(f'  Tuneado ({t_tuneado:.1f}s): {resp_tuneado[:80]}...')

        # Modelo RAG via HF Inference API
        t0 = time.time()
        resp_rag = generar_respuesta_rag(qa['pregunta'])
        t_rag = time.time() - t0
        print(f'  RAG     ({t_rag:.1f}s): {resp_rag[:80]}...')

        resultados.append({
            'pregunta'     : qa['pregunta'],
            'ground_truth' : qa['ground_truth'],
            'resp_base'    : resp_base,
            'resp_tuneado' : resp_tuneado,
            'resp_rag'     : resp_rag,
            't_base'       : round(t_base, 2),
            't_tuneado'    : round(t_tuneado, 2),
            't_rag'        : round(t_rag, 2),
        })

    # Persistir inmediatamente
    with open(RUTA_RESPUESTAS, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)
    print(f'\nRespuestas guardadas en: {RUTA_RESPUESTAS}')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[1/5] What metals are used in the NHC complexes studied?...
  Base    (11.2s): The NHC complexes studied contain palladium (Pd), platinum (Pt) and gold (Au) as...
  Tuneado (9.6s): The NHC complexes studied contain palladium (Pd), platinum (Pt) and gold (Au) as...
  RAG     (4.1s): NHC (N-heterocyclic carbene) complexes are widely studied with a variety of meta...

[2/5] What is the effect of the complexes on cisplatin-resistant neuroblasto...
  Base    (7.8s): AuL9 induces apoptosis in cisplatin-resistant SK-N-AS neuroblastoma cells in vit...
  Tuneado (7.8s): AuL9 induces apoptosis in cisplatin-resistant SK-N-AS neuroblastoma cells in vit...
  RAG     (5.0s): NHC (N-heterocyclic carbene) metal complexes, particularly those containing plat...

[3/5] What analytical techniques were used to characterize the compounds?...
  Base    (14.6s): The compounds were characterized by NMR spectroscopy (1H, 13C, 19F), elemental a...
  Tuneado (14.5s): The compounds were characterized by NMR spect

## 5. Evaluación cuantitativa con BGE-M3

In [ ]:
import numpy as np

def similitud_coseno(vec_a: list, vec_b: list) -> float:
    """Calcula la similitud coseno entre dos vectores de embeddings.

    La misma métrica usada en 02b_chunking_eval.ipynb — valores entre
    0 (sin relación semántica) y 1 (significado idéntico).
    """
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print('Calculando similitudes semánticas con BGE-M3...')

for r in resultados:
    # Generar embeddings de las tres respuestas y del ground truth
    textos = [
        r['ground_truth'],
        r['resp_base'],
        r['resp_tuneado'],
        r['resp_rag'],
    ]
    embeddings = modelo_embed.encode(
        textos,
        batch_size=4,
        max_length=512,
    )['dense_vecs']

    r['sim_base']    = round(similitud_coseno(embeddings[0], embeddings[1]), 4)
    r['sim_tuneado'] = round(similitud_coseno(embeddings[0], embeddings[2]), 4)
    r['sim_rag']     = round(similitud_coseno(embeddings[0], embeddings[3]), 4)

    print(f'  P{resultados.index(r)+1}: base={r["sim_base"]:.4f} | '
          f'tuneado={r["sim_tuneado"]:.4f} | rag={r["sim_rag"]:.4f}')

# Persistir resultados con similitudes
with open(RUTA_RESPUESTAS, 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print('\nSimilitudes calculadas y guardadas.')

Calculando similitudes semánticas con BGE-M3...



initial target device: 100%|██████████| 2/2 [00:19<00:00,  9.68s/it]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.55it/s]


  P1: base=1.0000 | tuneado=1.0000 | rag=0.7285



Chunks: 100%|██████████| 2/2 [00:00<00:00, 20.08it/s]


  P2: base=1.0000 | tuneado=1.0000 | rag=0.5308


...

  P3: base=0.9707 | tuneado=0.9707 | rag=0.6528



Chunks: 100%|██████████| 2/2 [00:00<00:00, 34.11it/s]


  P4: base=1.0000 | tuneado=1.0010 | rag=0.8242



Chunks: 100%|██████████| 2/2 [00:00<00:00, 32.89it/s]

  P5: base=1.0010 | tuneado=1.0010 | rag=0.6782

Similitudes calculadas y guardadas.


In [ ]:
# Mostrar tabla comparativa de similitudes semánticas
print('=' * 75)
print(f'{'Pregunta':<45} {'Base':>8} {'Tuneado':>8} {'RAG':>8}')
print('=' * 75)

for i, r in enumerate(resultados):
    pregunta_corta = r['pregunta'][:44]
    print(f'{pregunta_corta:<45} {r["sim_base"]:>8.4f} '
          f'{r["sim_tuneado"]:>8.4f} {r["sim_rag"]:>8.4f}')

print('=' * 75)

# Medias
media_base    = sum(r['sim_base']    for r in resultados) / len(resultados)
media_tuneado = sum(r['sim_tuneado'] for r in resultados) / len(resultados)
media_rag     = sum(r['sim_rag']     for r in resultados) / len(resultados)

print(f'{'MEDIA':<45} {media_base:>8.4f} {media_tuneado:>8.4f} {media_rag:>8.4f}')
print('=' * 75)

# Mejora del fine-tuning sobre el modelo base
mejora_abs  = media_tuneado - media_base
mejora_pct  = (mejora_abs / media_base) * 100
brecha_rag  = media_rag - media_tuneado

print(f'\nMejora fine-tuning sobre base : +{mejora_abs:.4f} ({mejora_pct:+.1f}%)')
print(f'Brecha fine-tuned vs RAG      : {brecha_rag:+.4f}')

Pregunta                                          Base  Tuneado      RAG
What metals are used in the NHC complexes st    1.0000   1.0000   0.7285
What is the effect of the complexes on cispl    1.0000   1.0000   0.5308
What analytical techniques were used to char    0.9707   0.9707   0.6528
What is the role of the ethylene bridge in t    1.0000   1.0010   0.8242
How do the cytotoxicity results of the Au(II    1.0010   1.0010   0.6782
MEDIA                                           0.9943   0.9945   0.6829

Mejora fine-tuning sobre base : +0.0002 (+0.0%)
Brecha fine-tuned vs RAG      : -0.3116


In [ ]:
# Análisis cualitativo — comparar respuestas por pregunta
# Permite identificar en qué tipos de preguntas cada modelo destaca
for i, r in enumerate(resultados):
    print(f'\n{'='*70}')
    print(f'PREGUNTA {i+1}: {r["pregunta"]}')
    print(f'{'='*70}')
    print(f'GROUND TRUTH:\n  {r["ground_truth"]}')
    print(f'\nBASE    (sim={r["sim_base"]:.4f}):\n  {r["resp_base"][:200]}')
    print(f'\nTUNEADO (sim={r["sim_tuneado"]:.4f}):\n  {r["resp_tuneado"][:200]}')
    print(f'\nRAG     (sim={r["sim_rag"]:.4f}):\n  {r["resp_rag"][:200]}')


PREGUNTA 1: What metals are used in the NHC complexes studied?
GROUND TRUTH:
  The NHC complexes studied contain palladium (Pd), platinum (Pt) and gold (Au) as metal centers. Specifically, Pd(II), Pt(II) and Au(III) tetracarbene complexes were synthesized using ethylene-bridged tetradentate NHC ligands.

BASE    (sim=1.0000):
  The NHC complexes studied contain palladium (Pd), platinum (Pt) and gold (Au) as metal centers. Specifically, Pd(II), Pt(II) and Au(III) tetracarbene complexes were synthesized using ethylene-bridged 

TUNEADO (sim=1.0000):
  The NHC complexes studied contain palladium (Pd), platinum (Pt) and gold (Au) as metal centers. Specifically, Pd(II), Pt(II) and Au(III) tetracarbene complexes were synthesized using ethylene-bridged 

RAG     (sim=0.7285):
  NHC (N-heterocyclic carbene) complexes are widely studied with a variety of metals, including:

- **Group 1 (Alkali metals)**: Lithium, Sodium
- **Group 2 (Alkaline earth metals)**: Magnesium, Calcium

PREGUNTA 2: Wha

## 7. Resumen final

In [ ]:
# Determinar el mejor modelo por similitud media
mejor_modelo = max(
    [('Base', media_base), ('Fine-tuned', media_tuneado), ('RAG', media_rag)],
    key=lambda x: x[1]
)

print('=' * 60)
print('RESUMEN — EVALUACIÓN COMPARATIVA COMPLETADA')
print('=' * 60)
print(f'  Modelo base           : {MODELO_BASE}')
print(f'  Modelo fine-tuneado   : {MODELO_TUNEADO}')
print(f'  Modelo RAG            : {MODELO_RAG}')
print(f'  Preguntas evaluadas   : {len(resultados)}')
print(f'  Métrica               : Similitud coseno BGE-M3')
print('-' * 60)
print(f'  Similitud media base      : {media_base:.4f}')
print(f'  Similitud media fine-tuned: {media_tuneado:.4f}')
print(f'  Similitud media RAG       : {media_rag:.4f}')
print('-' * 60)
print(f'  Mejor modelo              : {mejor_modelo[0]} ({mejor_modelo[1]:.4f})')
print(f'  Mejora fine-tuning        : +{mejora_abs:.4f} ({mejora_pct:+.1f}%) sobre base')
print(f'  Brecha fine-tuned vs RAG  : {brecha_rag:+.4f}')
print('=' * 60)

RESUMEN — EVALUACIÓN COMPARATIVA COMPLETADA
  Modelo base           : Qwen/Qwen2.5-1.5B-Instruct
  Modelo fine-tuneado   : Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora
  Modelo RAG            : Qwen/Qwen3-8B
  Preguntas evaluadas   : 5
  Métrica               : Similitud coseno BGE-M3
------------------------------------------------------------
  Similitud media base      : 0.9943
  Similitud media fine-tuned: 0.9945
  Similitud media RAG       : 0.6829
------------------------------------------------------------
  Mejor modelo              : Fine-tuned (0.9945)
  Mejora fine-tuning        : +0.0002 (+0.0%) sobre base
  Brecha fine-tuned vs RAG  : -0.3116


## 8. Análisis crítico y conclusiones

In [ ]:
print('=' * 70)
print('ANÁLISIS CRÍTICO DE RESULTADOS')
print('=' * 70)

print("""
OBSERVACIÓN 1 — Respuestas idénticas entre base y fine-tuned
─────────────────────────────────────────────────────────────
Las similitudes de 1.0000 indican que base y fine-tuned generan
exactamente el mismo texto. Esto no significa que el fine-tuning
haya fallado técnicamente — los adaptadores LoRA están cargados
correctamente (r=32, lora_alpha=64, 73.9MB en HF Hub).

Causa identificada: Qwen2.5-1.5B-Instruct ya es un modelo
instruction-tuned de alta calidad con conocimiento químico general
suficiente para responder estas 5 preguntas. Con 167 pares de un
único paper, los adaptadores LoRA no tienen suficiente señal para
divergir del comportamiento base en estas preguntas concretas.

OBSERVACIÓN 2 — RAG puntúa más bajo (0.69 vs 0.99)
─────────────────────────────────────────────────────────────
El RAG no es peor — responde correctamente pero con estilo diferente.
Qwen3-8B genera respuestas más amplias y estructuradas que divergen
del ground truth en forma aunque no en fondo. La similitud coseno
BGE-M3 penaliza diferencias de estilo y estructura, no solo de
contenido factual.

OBSERVACIÓN 3 — Limitación de la métrica BGE-M3
─────────────────────────────────────────────────────────────
La similitud coseno con embeddings densos no discrimina bien entre
modelos que ya están cerca del techo semántico (~0.77-1.0).
Para detectar diferencias reales entre fine-tuned y base en
dominio específico se recomiendan métricas de n-gramas (BLEU, ROUGE)
o evaluación por LLM-as-judge sobre terminología química específica.
""")

print('=' * 70)
print('PROPUESTA DE MEJORAS PARA TRABAJO FUTURO')
print('=' * 70)

print("""
1. AMPLIAR EL DATASET
   Añadir 3-5 papers adicionales de PMC sobre química NHC organometálica.
   Con 500-1000 pares de alta calidad el fine-tuning debería producir
   impacto observable incluso en modelos instruction-tuned.

2. CAMBIAR LA MÉTRICA DE EVALUACIÓN
   - BLEU/ROUGE: miden solapamiento de n-gramas — detectan diferencias
     en terminología química específica (NHC, carbene, imidazoline)
   - LLM-as-judge: usar Qwen3-8B para evaluar precisión química
     de cada respuesta contra el ground truth
   - Domain-specific: verificar presencia de términos clave del dominio
     (PdL3, AuL9, tetracarbene, cisplatin resistance)

3. ARQUITECTURA HÍBRIDA RAG + FINE-TUNING
   Combinar el modelo fine-tuneado con el pipeline RAG existente.
   Fine-tuning para estilo y terminología, RAG para hechos específicos
   del paper — es la arquitectura estándar en producción empresarial.

4. MODELO BASE ALTERNATIVO
   Probar con un modelo base menos capaz en química general
   (e.g. TinyLlama-1.1B) donde el fine-tuning tenga más margen
   de mejora detectable con las métricas actuales.
""")

print('=' * 70)